# TRÍ TUỆ NHÂN TẠO - THỰC HÀNH BUỔI 3
## THUẬT TOÁN DIJKSTRA TÌM ĐƯỜNG ĐI NGẮN NHẤT
**Họ và tên:** Nguyễn Trung Tín  
**MSSV:** 25110200091  
**Khoa:** Công nghệ Thông tin & Trí tuệ Nhân tạo (CNTT-TTNT)  
**Khóa:** 25  
**Môi trường:** Google Colab (Python 3)

In [ ]:
# Ô 1: thư viện và dữ liệu
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

VO_CUNG = math.inf

# Đồ thị vô hướng 6 đỉnh theo đề bài:
# SA=7, SB=2, AB=3, BC=6, AC=1, AD=4, CD=2, CT=7, DT=3
DO_THI = {
    "S": {"A": 7, "B": 2},
    "A": {"S": 7, "B": 3, "C": 1, "D": 4},
    "B": {"S": 2, "A": 3, "C": 6},
    "C": {"A": 1, "B": 6, "D": 2, "T": 7},
    "D": {"A": 4, "C": 2, "T": 3},
    "T": {"C": 7, "D": 3}
}

# Tọa độ các đỉnh (để vẽ hình giống như đề bài)
TOA_DO = {
    "S": (0.0, 0.0),
    "A": (2.4, 1.8),
    "B": (2.4, -1.8),
    "C": (5.2, 0.0),
    "D": (8.0, 1.8),
    "T": (10.6, 0.0)
}

NGUON = "S"

# Màu sắc theo bài giảng
MAU = {
    "den": "#1C1C1C",
    "xam": "#B0B0B0",
    "xanh": "#1F5C53",
    "xanh_nhat": "#DCEAE6",
    "cam": "#9A6414",
    "cam_nhat": "#F5E9D2",
    "do": "#A6432F"
}

def so(x):
    """In khoảng cách: vô cùng -> 'oo'."""
    return "oo" if x == VO_CUNG else str(x)
print("Khởi tạo dữ liệu đồ thị thành công! Nguồn:", NGUON)

In [ ]:
# Ô 2: thuật toán Dijkstra - mỗi bước đều ghi lời giải
def chup(buoc, hien_tai, dist, cha, da_xet, cap_nhat, loi_giai):
    """Lưu lại trạng thái sau một bước để vẽ hình."""
    return {
        "buoc": buoc,
        "hien_tai": hien_tai,
        "dist": dict(dist),
        "cha": dict(cha),
        "da_xet": list(da_xet),
        "cap_nhat": list(cap_nhat),
        "loi_giai": list(loi_giai)
    }

def dijkstra(do_thi, nguon):
    dist = {v: VO_CUNG for v in do_thi} # d[v]: khoảng cách tốt nhất đã biết
    cha = {v: None for v in do_thi}    # cha[v]: đỉnh đi trước v trên đường ngắn nhất
    da_xet = []                        # các đỉnh đã xét xong
    dist[nguon] = 0
    cac_buoc = [chup(0, None, dist, cha, da_xet, [],
                     ["Buoc 0 - Khoi tao",
                      "  d[%s] = 0 (tu %s den chinh no ton 0)" % (nguon, nguon),
                      "  cac dinh con lai d = oo (chua biet duong di)"])]
    dem = 0
    while True:
        # Việc 1: chọn đỉnh chưa xét có d nhỏ nhất
        ung_vien = [v for v in do_thi if v not in da_xet and dist[v] != VO_CUNG]
        if not ung_vien:
            break
        u = min(ung_vien, key=lambda v: dist[v])
        dem += 1
        loi = ["Buoc %d - Chon dinh %s vi d[%s] = %s la nho nhat trong cac dinh chua xet"
               % (dem, u, u, dist[u])]
        da_xet.append(u)
        
        # Việc 2: xét các đỉnh kề chưa xét, thử cập nhật
        cap_nhat = []
        for v, w in do_thi[u].items():
            if v in da_xet:
                continue
            moi = dist[u] + w
            if moi < dist[v]:
                loi.append("  Dinh ke %s: %s + %s = %s < %s -> CAP NHAT d[%s] = %s"
                           % (v, dist[u], w, moi, so(dist[v]), v, moi))
                dist[v] = moi
                cha[v] = u
                cap_nhat.append(v)
            else:
                loi.append("  Dinh ke %s: %s + %s = %s >= %s -> giu nguyen"
                           % (v, dist[u], w, moi, so(dist[v])))
        if not cap_nhat and len(loi) == 1:
            loi.append("  Khong con dinh ke nao chua xet -> khong co gi de cap nhat")
            
        # Việc 3: đánh dấu đã xét xong, lặp lại
        loi.append("  => Danh dau %s da xet xong" % u)
        cac_buoc.append(chup(dem, u, dist, cha, da_xet, cap_nhat, loi))
    return dist, cha, cac_buoc

def truy_vet(cha, dich):
    """Đi ngược theo cha từ đích về nguồn để lấy đường đi."""
    duong, v = [], dich
    while v is not None:
        duong.append(v)
        v = cha[v]
    return duong[::-1]
print("Đã định nghĩa xong thuật toán Dijkstra!")

In [ ]:
# Ô 3: vẽ hình có khung (3 khung: đồ thị, bảng, giải thích)
def khung(ax, tieu_de, mau):
    for s in ax.spines.values():
        s.set_edgecolor(mau)
        s.set_linewidth(2.2)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(tieu_de, fontsize=11, fontweight="bold", color=mau, loc="left")

def ve_do_thi(ax, b, do_thi, toa_do):
    nong = set()
    if b["hien_tai"] is not None:
        nong = {tuple(sorted((b["hien_tai"], v))) for v in b["cap_nhat"]}
    # Vẽ cạnh
    for u in do_thi:
        for v, w in do_thi[u].items():
            if u < v:
                (x1, y1), (x2, y2) = toa_do[u], toa_do[v]
                la_nong = (u, v) in nong
                ax.plot([x1, x2], [y1, y2], zorder=1, lw=5 if la_nong else 2,
                        color=MAU["cam"] if la_nong else MAU["xam"])
                ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(w), ha="center", va="center",
                        fontsize=11, zorder=2,
                        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none"))
    # Vẽ đỉnh
    for v, (x, y) in toa_do.items():
        if v == b["hien_tai"]:
            mn, vien, chu = MAU["do"], MAU["do"], "white" # đang xét
        elif v in b["da_xet"]:
            mn, vien, chu = MAU["xanh_nhat"], MAU["xanh"], MAU["den"] # đã xét
        elif v in b["cap_nhat"]:
            mn, vien, chu = MAU["cam_nhat"], MAU["cam"], MAU["den"] # vừa cập nhật
        else:
            mn, vien, chu = "white", MAU["den"], MAU["den"] # chưa xét
            
        ax.add_patch(Circle((x, y), 0.45, fc=mn, ec=vien, lw=2, zorder=3))
        ax.text(x, y, v, ha="center", va="center", fontsize=13, fontweight="bold", color=chu, zorder=4)
        ax.text(x, y + 0.75, "d=" + so(b["dist"][v]), ha="center", va="center", fontsize=10,
                zorder=4, bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=MAU["xam"]))
    
    xs = [p[0] for p in toa_do.values()]
    ys = [p[1] for p in toa_do.values()]
    ax.set_xlim(min(xs) - 1.2, max(xs) + 1.2)
    ax.set_ylim(min(ys) - 1.3, max(ys) + 1.6)
    ax.set_aspect("equal")

def ve_bang(ax, b):
    hang, mau_o = [], []
    for v in sorted(b["dist"]):
        if v == b["hien_tai"]:
            tt, m = "dang xet", "#F6E6E1"
        elif v in b["da_xet"]:
            tt, m = "da xet xong", MAU["xanh_nhat"]
        elif v in b["cap_nhat"]:
            tt, m = "vua cap nhat", MAU["cam_nhat"]
        else:
            tt, m = "chua xet", "white"
        hang.append([v, so(b["dist"][v]), b["cha"][v] or "-", tt])
        mau_o.append([m] * 4)
    t = ax.table(cellText=hang, colLabels=["Dinh", "d", "Cha", "Trang thai"],
                 cellColours=mau_o, loc="center", cellLoc="center")
    t.auto_set_font_size(False)
    t.set_fontsize(11)
    t.scale(1, 1.8)

def ve_buoc(b, do_thi=DO_THI, toa_do=TOA_DO):
    fig = plt.figure(figsize=(12, 7.2))
    gs = fig.add_gridspec(2, 2, width_ratios=[2, 1], height_ratios=[2.5, 1.1], hspace=0.2, wspace=0.12)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, :])
    khung(ax1, "KHUNG 1: Do thi (do=dang xet, xanh=da xet, cam=vua cap nhat)", MAU["xanh"])
    khung(ax2, "KHUNG 2: Bang d va cha", MAU["cam"])
    khung(ax3, "KHUNG 3: Giai thich buoc nay", MAU["do"])
    ve_do_thi(ax1, b, do_thi, toa_do)
    ve_bang(ax2, b)
    ax3.text(0.015, 0.92, "\n".join(b["loi_giai"]), va="top", ha="left", fontsize=11,
             family="monospace", transform=ax3.transAxes)
    plt.show()
print("Đã cấu hình xong hàm hiển thị trực quan đồ thị!")

In [ ]:
# Ô 4: chạy thuật toán, in lời giải và vẽ hình từng bước
dist, cha, cac_buoc = dijkstra(DO_THI, NGUON)
for b in cac_buoc:
    print("\n".join(b["loi_giai"]))
    print()
    ve_buoc(b)

In [ ]:
# Ô 5: kết quả cuối cùng và đường đi ngắn nhất
print("="*55)
print("KET QUA: khoang cach ngan nhat tu %s" % NGUON)
print("="*55)
for v in sorted(dist):
    if v == NGUON:
        continue
    print("  d(%s -> %s) = %2s  |  duong di: %s"
          % (NGUON, v, so(dist[v]), " -> ".join(truy_vet(cha, v))))
print("="*55)